# Grad-CAM Explainability for Alzheimer's Disease Classification

This notebook implements Gradient-weighted Class Activation Mapping (Grad-CAM) to show which regions of an MRI slice drive the VGG16-DCNN prediction. The gradients of the class score with respect to the final convolutional feature maps (`block5_conv3`) are averaged to obtain importance weights. Their weighted sum, passed through ReLU, gives the localisation map, which is up-sampled and overlaid on the MRI image.

## Prerequisites

- Python 3.10+
- Required libraries: `numpy`, `opencv-python`, `pandas`, `matplotlib`, `tensorflow`
- A trained model from `VGG16_DCNN_classification.ipynb`

## Workflow

1. Load the trained VGG16-DCNN model
2. Compute the gradients of the class score with respect to the `block5_conv3` feature maps
3. Average the gradients to obtain the feature-map weights α_k^c
4. Compute the Grad-CAM map L^c = ReLU(Σ_k α_k^c F_k)
5. Up-sample the map and overlay it on the MRI image

In [ ]:
# ==========================================================
# IMPORTS & CONFIGURATION
# ==========================================================

import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input

MODEL_PATH = "./outputs/06_models/vgg16_dcnn_fold0.keras"
TEST_CSV = "./outputs/04_segmented/test_segmented.csv"
GRADCAM_DIR = "./outputs/08_gradcam"
CLASS_NAMES = ["MD", "MOD", "ND", "VMD"]
NUM_CLASSES = 4
IMG_SIZE = 224
LAST_CONV_LAYER = "block5_conv3"
IMAGES_PER_CLASS = 3
RANDOM_SEED = 42

model = tf.keras.models.load_model(MODEL_PATH)

## Load ROI Image

Applies the same input normalisation used during training.

In [ ]:
# ==========================================================
# LOAD ROI
# ==========================================================

def load_roi(path, label=None):
    """Read ROI -> resize 224x224 -> grayscale to 3 channels -> VGG16 normalisation."""
    img = tf.io.decode_png(tf.io.read_file(path), channels=1)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = preprocess_input(tf.image.grayscale_to_rgb(img))
    if label is None:
        return img
    return img, tf.one_hot(label, NUM_CLASSES)

## Grad-CAM Map

Computes the class-specific importance weights and the localisation map from the final convolutional layer.

In [ ]:
# ==========================================================
# GRAD-CAM
# ==========================================================

def grad_cam(model, img_tensor, class_idx=None):
    grad_model = tf.keras.Model(model.inputs, [model.get_layer(LAST_CONV_LAYER).output, model.output])
    with tf.GradientTape() as tape:
        feature_maps, preds = grad_model(img_tensor[None, ...], training=False)
        if class_idx is None:
            class_idx = int(tf.argmax(preds[0]))
        y_c = preds[:, class_idx]
    grads = tape.gradient(y_c, feature_maps)
    alpha = tf.reduce_mean(grads, axis=(1, 2))                                      # alpha_k^c
    cam = tf.nn.relu(tf.reduce_sum(feature_maps * alpha[:, None, None, :], axis=-1))[0]
    cam = cam.numpy() / (cam.numpy().max() + 1e-8)
    return cam, class_idx, preds.numpy()[0]


def overlay(gray, cam, alpha=0.4):
    cam = cv2.resize(cam, gray.shape[::-1])
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    return cv2.addWeighted(heat, alpha, cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR), 1 - alpha, 0)

## Visualise Grad-CAM for Test Images

Generates Grad-CAM overlays for sample test images of each class and saves them.

In [ ]:
# ==========================================================
# VISUALISE GRAD-CAM
# ==========================================================

os.makedirs(GRADCAM_DIR, exist_ok=True)
test_df = pd.read_csv(TEST_CSV)
samples = test_df.sample(frac=1.0, random_state=RANDOM_SEED).groupby("label").head(IMAGES_PER_CLASS)

for _, r in samples.iterrows():
    cam, cls, probs = grad_cam(model, load_roi(r["roi_path"]))
    gray = cv2.imread(r["roi_path"], cv2.IMREAD_GRAYSCALE)
    vis = overlay(gray, cam)
    cv2.imwrite(os.path.join(GRADCAM_DIR, f"gradcam_{r['label']}_{os.path.basename(r['roi_path'])}"), vis)

    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(gray, cmap="gray"); ax[0].set_title(f"ROI (true: {r['label']})")
    ax[1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax[1].set_title(f"Grad-CAM (pred: {CLASS_NAMES[cls]}, {probs[cls] * 100:.1f}%)")
    for a in ax:
        a.axis("off")
    plt.tight_layout(); plt.show()